<a href="https://colab.research.google.com/github/SheffieldML/GPc/blob/main/python/examples/mlp_olympic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Olympic marathon: RBF vs Williams MLP (arcsin)

CIP-001D Phase F. Same data, two priors — squared-exponential vs the infinite-width MLP / arcsin kernel (Williams 1996).

```bash
pip install pods matplotlib   # or: pip install 'gpc[examples]'
```


In [ ]:
%pip install --no-cache-dir --upgrade --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "gpc==0.1.0a3"

In [ ]:
%pip install pods matplotlib


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import sys
from pathlib import Path

root = Path.cwd()
if (root / "gpc").is_dir():
    sys.path.insert(0, str(root))
elif (root.parent / "gpc").is_dir():
    sys.path.insert(0, str(root.parent))

import numpy as np
import gpc

try:
    import pods
except ImportError as exc:
    raise SystemExit("pip install pods  (or gpc[examples])") from exc

data = pods.datasets.olympic_marathon_men()
X = np.asarray(data["X"], dtype=float)
Y = np.asarray(data["Y"], dtype=float)
if X.ndim == 1:
    X = X[:, None]
if Y.ndim == 1:
    Y = Y[:, None]

m_rbf = gpc.GPRegression(X, Y, kernel="rbf")
m_rbf.set_by_path("rbf.lengthscale", 30.0)
m_mlp = gpc.GPRegression(X, Y, kernel="mlp")

for m in (m_rbf, m_mlp):
    m.optimize(iters=100)

print("RBF objective:", m_rbf.objective())
print("MLP objective:", m_mlp.objective())


In [ ]:
from gpc.plot import plot_1d
import matplotlib.pyplot as plt

Xtest = np.linspace(float(X.min()) - 4.0, float(X.max()) + 4.0, 400)[:, None]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
plot_1d(m_rbf, X, Y, Xtest=Xtest, ax=axes[0], label="RBF")
axes[0].set_title("RBF / SE")
axes[0].set_xlabel("year")
axes[0].set_ylabel("time (hours)")
plot_1d(m_mlp, X, Y, Xtest=Xtest, ax=axes[1], label="MLP (arcsin)")
axes[1].set_title("Williams MLP / arcsin")
axes[1].set_xlabel("year")
fig.suptitle("Olympic marathon — same data, different prior geometry")
plt.tight_layout()
plt.show()
